# Bước 1: PaddleOCR Detection Stage Demo

Notebook này minh họa cách sử dụng module `TextDetector` (được tích hợp với pre-trained DBNet của **PaddleOCR**) để phát hiện và cắt (crop) các vùng chứa văn bản từ tài liệu PDF (hoặc ảnh) đấu giá xe.

In [ ]:
import sys
import os
import cv2
import matplotlib.pyplot as plt

# Đưa thư mục gốc của project vào path để import module
sys.path.append(os.path.abspath('..'))

from src.hybrid_ocr.detection.detect import TextDetector, render_pdf_pages

# Hiển thị plot to hơn một chút
plt.rcParams['figure.figsize'] = (12, 12)

## 1. Trích xuất trang PDF thành Ảnh
Trước tiên, chúng ta sẽ kết xuất (render) trang đầu tiên của file PDF mẫu thành định dạng ảnh.

In [ ]:
pdf_path = "../pdfs/2026年6月25日-JU愛知-2163-通常車-151-200.pdf"
tmp_img_dir = "../scratch/demo_pages"

os.makedirs(tmp_img_dir, exist_ok=True)

# Chỉ render trang đầu tiên (page 0) với độ phân giải DPI = 200
print("Đang render PDF thành ảnh...")
image_paths = render_pdf_pages(pdf_path, tmp_img_dir, dpi=200, page_range=(0, 0))
sample_image_path = image_paths[0]
print(f"Đã lưu trang PDF thành ảnh tại: {sample_image_path}")

## 2. Khởi tạo PaddleOCR Text Detector
Khởi tạo bộ detector DBNet đã được huấn luyện sẵn của PaddleOCR tối ưu cho tài liệu văn bản tiếng Nhật.

In [ ]:
detector = TextDetector(
    confidence_threshold=0.3, 
    device="auto" # Tự động chọn CPU/MPS/CUDA
)

print("Đã khởi tạo PaddleOCR TextDetector thành công!")

## 3. Chạy Phát hiện (Detection) và Cắt vùng (Crop)
Chúng ta sẽ đưa ảnh vào detector. Module sẽ trả về tọa độ bbox, độ tự tin (confidence), và ảnh crop của từng vùng chữ đã được xoay phẳng (deskewed).

In [ ]:
# Đọc ảnh bằng OpenCV (OpenCV dùng BGR)
img = cv2.imread(sample_image_path)

# Chạy detect
print("Đang chạy PaddleOCR detection...")
detections = detector.detect(
    img, 
    return_crops=True, 
    apply_deskew=True,
    crop_padding=5
)

print(f"Phát hiện được {len(detections)} vùng chứa văn bản.")

## 4. Trực quan hóa kết quả (Visualization)
Vẽ các bounding boxes màu xanh lên ảnh gốc để xem PaddleOCR đã khoanh vùng chính xác các dòng chữ ở mọi nơi như thế nào.

In [ ]:
# Tạo ảnh trực quan (vẽ box)
vis_img = detector.visualize(img, detections)

# Chuyển từ BGR sang RGB để vẽ bằng Matplotlib
vis_img_rgb = cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(15, 20))
plt.imshow(vis_img_rgb)
plt.title("Kết quả PaddleOCR Detection trên tài liệu")
plt.axis('off')
plt.show()

## 5. Xem trước các vùng ảnh đã được Crop (Ảnh đầu vào cho Bước 2)
Hiển thị ngẫu nhiên một số ảnh chữ nhỏ đã được crop và xoay phẳng để chuẩn bị nhận dạng.

In [ ]:
num_crops_to_show = min(10, len(detections))

if num_crops_to_show > 0:
    fig, axes = plt.subplots(num_crops_to_show, 1, figsize=(6, 2 * num_crops_to_show))
    
    # Nếu chỉ có 1 ảnh, subplot không trả về array
    if num_crops_to_show == 1:
        axes = [axes]
        
    for i in range(num_crops_to_show):
        det = detections[i]
        crop_rgb = cv2.cvtColor(det["crop"], cv2.COLOR_BGR2RGB)
        
        axes[i].imshow(crop_rgb)
        axes[i].set_title(f"BBox: {det['bbox']}")
        axes[i].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy bounding box nào để crop.")